## Further exploration: finding an audio input

the first pass (`00` to `03`) only ever traced a visual path, `ol_sensory` -> `vnc_motor`. this file asks whether audio input exists in the data at all, then builds a path from a hearing neuron toward motor output, by hand this time

along the way, it goes back and checks both ends of the visual path, which turns up a mistake worth knowing about

### Init

In [37]:
import pandas as pd
from pathlib import Path
import pyarrow.dataset as ds
import pyarrow.compute as pc

root = Path.cwd().parent
# root = Path(__file__).resolve().parent.parent

neurons_file = root / 'data' / 'body-annotations-male-cns-v1.0-minconf-0.5.feather'
connections_file = root / 'data' / 'connectome-weights-male-cns-v1.0-minconf-0.5.feather'

df = pd.read_feather(neurons_file)
traced = df[df['status'] == 'Traced']

# more for building
id_to_superclass = traced.set_index('bodyId')['superclass'].to_dict()

# lazy loading, the connections file is too big to load in full
dataset = ds.dataset(connections_file, format="feather")

#### Does audio input exist in this data?

in the annotations cheatsheet, `cb_sensory` is the superclass for sensory input that doesn't come through the vnc, which could possibly include hearing. although superclass only says where in the body a neuron lives, and it doesn't say what it senses. there are two columns that could indicate that:
- `receptorType`: what kind of sense the neuron detects, so this could be used to separate hearing from smell or touch
- `entryNerve`: the way in for what you feel, the named cable this sensory neuron's wire uses to enter the brain or nerve cord from the body

<div align="center">
  <img src="../img/1.png" alt="entry and exit nerve example" width="40%">
</div>

A few notes on reading the figure:
- Each node's tag is just the values from that row. Only the first node has entryNerve filled in, and only the last has exitNerve.
- The two middle neurons have both columns empty, and so do most neurons in the dataset.
- The amber sleeves are the stretch of the first and last neurons' own fibers that runs outside the brain and nerve cord, which is all the cable is.

In [38]:
print(traced[['superclass', 'entryNerve']].value_counts().sort_index().head(10))

print("cb_sensory data ----------------")

cb_sensory = traced[traced['superclass'] == 'cb_sensory']

print(cb_sensory['receptorType'].value_counts())
print(cb_sensory['entryNerve'].value_counts())


superclass         entryNerve
cb_sensory         AN            3315
                   MxLbN         1387
                   ON              33
                   PhN             46
                   aPhN            78
cb_sensory_tbc     ON               2
sensory_ascending  ADMN            83
                   AbN2             6
                   AbN3             2
                   AbN4             8
Name: count, dtype: int64
cb_sensory data ----------------
Series([], Name: count, dtype: int64)
entryNerve
AN       3315
MxLbN    1387
aPhN       78
PhN        46
ON         33
Name: count, dtype: int64


> the first print with the traced dataset is saying: `cb_sensory` neurons come in through five different cables: `AN` (3315), `MxLbN` (1387), `aPhN` (78), `PhN` (46) and `ON` (33), so most of them come in through `AN`, while `cb_sensory_tbc`, the provisional label, has 2 neurons, both in `ON`
>
> `receptorType` came back empty for `cb_sensory`, and `entryNerve` did come back

In [39]:
# is receptorType populated anywhere or is it just for cb_sensory?
print(traced[['superclass', 'receptorType']].value_counts())

superclass   receptorType  
vnc_sensory  putative_ppk23    269
             putative_ppk25    257
             putative_IR52b    226
Name: count, dtype: int64


> `receptorType` isn't broken, it's just populated for `vnc_sensory`, not `cb_sensory`
>
> the values are `ppk23`, `ppk25`, `IR52b`, genes that play roles in pheromone detection, species recognition, and the regulation of courtship behaviors
>
> conclusion: receptor types may not be useful to look at

#### The antennal nerve

`entryNerve` did come back, and `AN` has a large neuron count. `AN` is assumed to be the antennal nerve (from the naming convention, not checked against documentation), and the antenna carries both hearing and smell, so these could be hearing, smell, or a mix of both

check whether `class` or `type` says which sense these neurons are for, since `receptorType` doesn't

In [40]:
an_neurons = traced[traced['entryNerve'] == 'AN']
print(an_neurons['class'].value_counts())

class
olfactory             2455
mechanosensory         744
hygrosensory            66
thermosensory           25
unknown_sensory         22
mechanosensory_tbc       1
Name: count, dtype: int64


> `class` confirms both are there: 2455 `olfactory` and 744 `mechanosensory`
>
> mechanosensory means systems that can respond to sound, touch, motion, etc

In [41]:
# finer-grained follow-up
print(an_neurons['type'].value_counts().head(20))

type
ORN_DA1       204
ORN_VA1d      132
ORN_VA1v      130
ORN_DL3       103
JO-unclear    102
ORN_VL2a       98
ORN_VM5d       84
ORN_VA2        83
ORN_DL1        83
ORN_VL1        82
ORN_VM4        78
ORN_DM1        74
JO-FV          70
BM             69
ORN_VA6        63
ORN_DM3        63
ORN_DL4        62
ORN_DM6        58
ORN_V          55
ORN_DM2        54
Name: count, dtype: int64


> `ORN_*` types dominate (olfactory receptor neurons), matching the fact that the `olfactory` class dominates
>
> a few `JO-*` types also show up, Johnston's organ on the antenna, responsible for hearing and sound detection

#### Johnston's organ (JO)

In [42]:
# drill down on the JO types
jo_neurons = traced[traced['type'].str.contains('JO', na=False)] # na values get treated as false instead of returning nan
print(jo_neurons['type'].value_counts())
print(jo_neurons.shape)

type
JO-unclear      102
JO-FV            70
JO-EV1           50
JO-EV3           45
JO-ED2_a         38
JO-EV2           36
JO-CM            30
JO-A-unclear     24
JO-EV5           23
JO-EV6           21
JO-ED2_b         21
JO-CL            19
JO-ED1           18
JO-B1_a          17
JO-ED2_c         14
JO-A2            13
JO-B-unclear     13
JO-B3            12
JO-B4_b          12
JO-B1_b          12
JO-mz            11
JO-B1_c          10
JO-CA1           10
JO-CA2            9
JO-B2             9
JO-A1             8
JO-DA             5
JO-FD1            4
JO-FD2            4
JO-B4_a           3
JO-DP             3
JO-A4             3
JO-A3             2
JO-EV4            1
Name: count, dtype: int64
(672, 36)


> there are a lot of JO families, maybe for different purposes -> can research and look into those specifics

in the published literature, JO is split into 5 subgroups by projection pattern in the brain: JO-A through JO-E

JO-A and JO-B are the established sound/vibration sensors
- JO-A: high frequency tuned, connects strongly to the escape pathway
- JO-B: low frequency tuned, connects to both escape AND the song-relay pathway (song-relay is fly courtship song processing, closest match to "rhythm")

JO-C and JO-E respond to static deflection, like gravity and wind. JO-D responds to both vibration and deflection, so it may not be wind/gravity only

the data also has `JO-F*` types (`JO-FV`, `JO-FD1`, `JO-FD2`, 78 neurons) that the sources above don't cover, so what they sense isn't known

sources:
- Ishikawa et al. 2020 (frontiersin.org/articles/10.3389/fphys.2019.01552)
- Mamiya lab et al., wiring patterns paper (pmc.ncbi.nlm.nih.gov/articles/PMC7676477)

In [43]:
jo_auditory = jo_neurons[jo_neurons['type'].str.contains('JO-A|JO-B', na=False, regex=True)]
print(jo_auditory['type'].value_counts())
print(jo_auditory.shape)

type
JO-A-unclear    24
JO-B1_a         17
JO-A2           13
JO-B-unclear    13
JO-B3           12
JO-B4_b         12
JO-B1_b         12
JO-B1_c         10
JO-B2            9
JO-A1            8
JO-B4_a          3
JO-A4            3
JO-A3            2
Name: count, dtype: int64
(138, 36)


> JO-B outnumbers JO-A, 88 versus 50, once you sum each family's subtypes
>
> combined with the finding that JO-B is the one wired toward song-relay, and has more neurons, JO-B is a strong candidate for a rhythm-based input stage

#### Side track: checking both ends of the visual path

in the initial exploration, it was concluded that an image had to go from `ol_sensory` -> `vnc_motor`. there was a gap in that, revealed by the exercise above: the subgroups of `ol_sensory` and `vnc_motor` were never checked, so it isn't known whether the correct responsible neurons were used

checking the start first:

In [44]:
ol_sensory_neurons = traced[traced['superclass'] == 'ol_sensory']
print(ol_sensory_neurons['type'].value_counts())
print(ol_sensory_neurons.shape)

type
R1-R6           1394
R8y              481
R7y              481
R8_unclear       442
R7_unclear       404
R7p              332
R8p              330
R7R8_unclear      85
R7d               82
R8d               76
HBeyelet           7
Name: count, dtype: int64
(4114, 36)


> by sheer luck, the random neuron chosen for `ol_sensory` was of type `R1-R6`, which is responsible for motion and brightness detection, more appropriate for reacting to a moving visual pattern
>
> `R7`/`R8` (color) and `HBeyelet` (probably the extraretinal eyelet, not used for image-forming vision) sit in the same superclass, so the pick could have gone another way

now the other end, `vnc_motor`:

In [45]:
vnc_motor_neurons = traced[traced['superclass'] == 'vnc_motor']
print(vnc_motor_neurons['type'].value_counts())
print(vnc_motor_neurons.shape)
print()
print(vnc_motor_neurons['exitNerve'].value_counts())
print()
print(vnc_motor_neurons['somaNeuromere'].value_counts())

type
Acc. ti flexor MN               47
Ti flexor MN                    37
Tr flexor MN                    35
Sternal posterior rotator MN    22
Acc. tr flexor MN               22
                                ..
MNml77                           2
MNhl60                           2
MNml83                           1
MNad50                           1
MNml76                           1
Name: count, Length: 142, dtype: int64
(708, 36)

exitNerve
MetaLN    122
MesoLN    116
ProLN      81
AbN4       72
AbNT       64
AbN2       44
AbN1       30
ADMN       28
DProN      28
AbN3       28
ProAN      24
VProN      22
MesoAN     19
PDMNa      16
PDMNp       6
CvN         4
DMetaN      4
Name: count, dtype: int64

somaNeuromere
T2     175
T1     173
T3     152
A1      56
A2      28
A4      22
A3      22
A9      21
A8      18
A5      16
A10     10
A6       8
A7       6
Name: count, dtype: int64


> `somaNeuromere` splits `vnc_motor` into thoracic (`T1`/`T2`/`T3`, where the legs attach, 500 of 708) and abdominal (`A1` to `A10`)
>
> `exitNerve` points the same way: `ProLN`/`MesoLN`/`MetaLN` (read as leg nerves) versus `AbN*` (read as abdominal nerves), both from the naming

check the end node of the path traced in `03_scaled_lif.ipynb`:

In [46]:
print(vnc_motor_neurons[vnc_motor_neurons['bodyId'] == 164190][['type', 'exitNerve', 'somaNeuromere']])

          type exitNerve somaNeuromere
116318  MNad21      AbN2            A1


> `164190` is `MNad21`, `exitNerve` `AbN2`, `somaNeuromere` `A1`
>
> `Ab` almost certainly stands for abdominal, and `A1` is the first abdominal segment, so the path traced in `03` ends at an abdominal motor neuron, not a leg one
>
> next time, check both the start and end points for where they actually are

this would help find a leg motor instead (needs `G` and `source_id` from `02_graph.ipynb`, so it's left commented out here):

In [47]:
# leg_motor_ids = vnc_motor_neurons[vnc_motor_neurons['somaNeuromere'].isin(['T1', 'T2', 'T3'])]['bodyId']
# valid_targets = [nid for nid in leg_motor_ids if nid in G.nodes]
# print(len(valid_targets))

# reachable = nx.descendants(G, source_id)
# reachable_motor = [nid for nid in valid_targets if nid in reachable]
# print(len(reachable_motor))

#### Back to the JO neurons: building a path by hand

returning to the JO neurons, the next step is to build a path just like `ol_sensory` -> `vnc_motor`, except this time by hand

`networkx` can find a shortest path between two IDs in one line, but it's not known which two IDs matter, there is no confirmed "sound goes in here" neuron and "movement comes out here" neuron to connect. thus hopping manually is done

In [48]:
# prints all columns - to look for any JO-B* neurons
with pd.option_context('display.max_columns', None):
    print(jo_auditory.head())

jo_id = jo_auditory['bodyId'].iloc[3] # with the above, it was found to be here
print('bodyId we are interested in:', jo_id)

       assignedOlHex1  assignedOlHex2  bodyId flywireType  group   instance  \
13927             NaN             NaN   25108       JO-A2    NaN    JO-A2_L   
17946             NaN             NaN   29561       JO-A2    NaN    JO-A2_L   
23125             NaN             NaN   35351       JO-A1    NaN    JO-A1_L   
29653             NaN             NaN   42619     JO-B1_a    NaN  JO-B1_a_L   
32722             NaN             NaN   46071       JO-A2    NaN    JO-A2_L   

      somaSide       statusLabel  superclass     type         vfbId  \
13927      NaN  RT Hard to trace  cb_sensory    JO-A2  VFB_jrmc2hkm   
17946      NaN  RT Hard to trace  cb_sensory    JO-A2  VFB_jrmc2hkr   
23125      NaN  RT Hard to trace  cb_sensory    JO-A1  VFB_jrmc2hks   
29653      NaN  RT Hard to trace  cb_sensory  JO-B1_a  VFB_jrmc2hlf   
32722      NaN  RT Hard to trace  cb_sensory    JO-A2  VFB_jrmc2hkn   

      hemibrainType itoleeHl supertype birthtime  mancBodyid  mancGroup  \
13927           NaN    

> lands on `JO-B1_a` (`bodyId` 42619), a fully-resolved `JO-B` type rather than one of the `unclear` rows

#### Hop 1: neighbours of the JO neuron

what neurons are outbound from the chosen `JO-B*` one? same neighbour-finding move as before

In [49]:
filter_expr = (pc.field("body_pre") == jo_id) | (pc.field("body_post") == jo_id)
jo_neighbours = dataset.to_table(filter=filter_expr).to_pandas()
print(jo_neighbours.shape)
print(jo_neighbours)

(285, 3)
      body_pre  body_post  weight
0        42619      15233      46
1        42619     525475      45
2        42619      13014      42
3        42619      11378      42
4        42619      12066      39
..         ...        ...     ...
280  915820880      42619       1
281      10074      42619       1
282  923995712      42619       1
283  941637235      42619       1
284  968187269      42619       1

[285 rows x 3 columns]


> some `body_pre`/`body_post` values are enormous, same pattern as with `11139`

In [50]:
# same move as with 11139 and ol_sensory,
# translate through id_to_superclass before trusting any of these IDs as meaningful
jo_neighbours['pre_type'] = jo_neighbours['body_pre'].map(id_to_superclass)
jo_neighbours['post_type'] = jo_neighbours['body_post'].map(id_to_superclass)

# dropna=False matters here, want to see how many of those 285 rows map to NaN
# Tells pandas to treat NaN as its own separate group or category rather than ignoring it
print(jo_neighbours['pre_type'].value_counts(dropna=False))
print()
print(jo_neighbours['post_type'].value_counts(dropna=False))

pre_type
cb_sensory          230
cb_intrinsic         26
NaN                  25
ascending_neuron      4
Name: count, dtype: int64

post_type
NaN                   177
cb_sensory             56
cb_intrinsic           47
ascending_neuron        3
descending_neuron       1
visual_centrifugal      1
Name: count, dtype: int64


In [51]:
# the strongest connections are the ones most likely to matter for a "does this respond to rhythmic input" story
top_outgoing = jo_neighbours[jo_neighbours['body_pre'] == jo_id].sort_values('weight', ascending=False)
print(top_outgoing.head(10))

    body_pre  body_post  weight    pre_type     post_type
0      42619      15233      46  cb_sensory  cb_intrinsic
1      42619     525475      45  cb_sensory  cb_intrinsic
2      42619      13014      42  cb_sensory  cb_intrinsic
3      42619      11378      42  cb_sensory  cb_intrinsic
4      42619      12066      39  cb_sensory  cb_intrinsic
5      42619      11275      35  cb_sensory  cb_intrinsic
6      42619      13546      28  cb_sensory  cb_intrinsic
8      42619      13619      25  cb_sensory  cb_intrinsic
9      42619      12530      25  cb_sensory  cb_intrinsic
11     42619      10795      17  cb_sensory  cb_intrinsic


> `JO-B1_a` itself classifies as `cb_sensory`, already central-brain sensory (`AN` carries straight to the brain)
>
> the strongest connections are all from the sensory neuron to central brain processing neurons, and it's quite consistent, unlike `ol_sensory`, which went to different superclasses within its top 10 weights
>
> however, of the 285 neighbour rows, 177 had no superclass (NaN), so most of the neighbours were unlabelled.

The top 10 shows only the ten strongest connections. In the data they all went to cb_intrinsic, so it looks like this neuron talks to cb_intrinsic and nothing else.

If each of those 177 unlabelled neighbours had a weight of 15. They'd add up to 177 * 15 = 2655, far more than cb_intrinsic's 490, even though none would show up in the top 10. Then claiming the neuron "mainly goes to cb_intrinsic" would have been wrong

So check the total weight of each superclass:

In [52]:
weighted_targets = (
    jo_neighbours[jo_neighbours['body_pre'] == jo_id]
    .groupby('post_type', dropna=False)['weight']
    .sum()
    .sort_values(ascending=False)
)
# returns a pandas series, not a set
# The outer parentheses don't create anything. They just split a long chain across several lines.
print(weighted_targets)

post_type
cb_intrinsic          490
NaN                   302
ascending_neuron        5
descending_neuron       5
visual_centrifugal      1
Name: weight, dtype: int64


> the 177 NaN rows sum to only 302 weight, compared to `cb_intrinsic` at 490

Further checking how heavy the weights of the NaN values are:

In [53]:
nan_targets = jo_neighbours[
    (jo_neighbours['body_pre'] == jo_id) & (jo_neighbours['post_type'].isna())
]
print(nan_targets.sort_values('weight', ascending=False))

     body_pre  body_post  weight    pre_type post_type
21      42619  381005967       9  cb_sensory       NaN
24      42619     388833       7  cb_sensory       NaN
28      42619  935548232       6  cb_sensory       NaN
29      42619  970439202       6  cb_sensory       NaN
36      42619  851972190       5  cb_sensory       NaN
..        ...        ...     ...         ...       ...
232     42619  107402883       1  cb_sensory       NaN
233     42619  108396158       1  cb_sensory       NaN
234     42619  124879930       1  cb_sensory       NaN
235     42619  128731924       1  cb_sensory       NaN
236     42619  135499287       1  cb_sensory       NaN

[177 rows x 5 columns]


> the NaN weights are all tiny (9 down to 1), showing a lot of weak connections

New question: why do those neurons have no superclass, are they real neurons that are missing, or are they just not annotated at all?
- "Real neurons we're missing" would mean the lookup is dropping something it shouldn't -> filter problem
- "Not annotated at all" would mean ignoring them is safe -> simply a data problem

In [54]:
# for further testing:
nan_ids = nan_targets['body_post'].unique()
print(df[df['bodyId'].isin(nan_ids)][['bodyId', 'status', 'statusLabel']])

# further confirming it
print(len(nan_ids), 'unique nan targets')
print(df['bodyId'].isin(nan_ids).sum(), 'of those exist anywhere in df')

           bodyId  status       statusLabel
157265     527138  Traced  RT Hard to trace
198806  541246868  Orphan            Orphan
177 unique nan targets
2 of those exist anywhere in df


> only 2 came back, so many of the connections supposedly exist but are not annotated
>
> sanity check: 177 unique NaN targets, and only 2 of them exist in the annotation table at all, `527138` (Traced, but no superclass filled in) and `541246868` (an orphan)
>
> so the other 175 aren't annotated at all, and at most one real neuron (`527138`) has no superclass, in conclusion, ignoring the NaN targets is safe

In [55]:
# 527138 is from above, it's weird because it's traced but still empty
print(traced[traced['bodyId'] == 527138][['bodyId', 'status', 'statusLabel', 'superclass', 'class', 'type']])

        bodyId  status       statusLabel superclass class type
157265  527138  Traced  RT Hard to trace        NaN   NaN  NaN


> otherwise: `JO-B1_a`'s wiring has a strong, concentrated signal into `cb_intrinsic`, with a long tail of weak connections mostly landing on unannotated fragments, and one lone Traced-but-unclassified neuron sitting in that tail

#### Hop 2: the strongest target

the natural next hop: pick the single strongest target, `body_post` 15233, weight 46

In [56]:
next_id = 15233
filter_expr = (pc.field("body_pre") == next_id) | (pc.field("body_post") == next_id)
next_neighbours = dataset.to_table(filter=filter_expr).to_pandas()
next_neighbours['pre_type'] = next_neighbours['body_pre'].map(id_to_superclass)
next_neighbours['post_type'] = next_neighbours['body_post'].map(id_to_superclass)
print(next_neighbours.shape)
print(next_neighbours['post_type'].value_counts(dropna=False))

(1423, 5)
post_type
NaN                  975
cb_intrinsic         435
descending_neuron      5
ascending_neuron       5
cb_sensory             3
Name: count, dtype: int64


> `(1423, 5)`, a lot more neighbours
>
> there are 5 descending neurons in there, which may be the bridge from brain to vnc

next, filter to outgoing only before asking where its signal goes

In [57]:
outgoing = next_neighbours[next_neighbours['body_pre'] == next_id]
print(outgoing.shape)
print(outgoing['post_type'].value_counts(dropna=False))

weighted_out = (
    outgoing.groupby('post_type', dropna=False)['weight']
    .sum()
    .sort_values(ascending=False)
)
print(weighted_out)

(1139, 5)
post_type
NaN                  975
cb_intrinsic         151
descending_neuron      5
ascending_neuron       5
cb_sensory             3
Name: count, dtype: int64
post_type
NaN                  1297
cb_intrinsic         1148
descending_neuron      42
ascending_neuron       17
cb_sensory              3
Name: weight, dtype: int64


> NaN bucket is large, so the same check from hop 1 is needed:

In [58]:
nan_out = outgoing[outgoing['post_type'].isna()]
print(nan_out['weight'].describe())

nan_ids = nan_out['body_post'].unique()
print(len(nan_ids), 'unique nan targets')
print(df['bodyId'].isin(nan_ids).sum(), 'of those exist anywhere in df')

count    975.000000
mean       1.330256
std        0.688463
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        7.000000
Name: weight, dtype: float64
975 unique nan targets
2 of those exist anywhere in df


> the NaN bucket is 975 unique targets with weights of 1 to 7 (median 1), and only 2 of them are in the annotation table at all, so it's the same unannotated-fragment pattern as hop 1, and no strong connection is hidden in there
>
> the 1297 NaN weight is large only because there are so many of them, so `cb_intrinsic` is still the real strongest destination, and the descending neuron link (42 of 2507) is still a trickle, not a highway, right now. it's worth knowing precisely before deciding whether this is a dead end or just a narrow doorway:

In [59]:
descending_out = outgoing[outgoing['post_type'] == 'descending_neuron'].sort_values('weight', ascending=False)
print(descending_out)

      body_pre  body_post  weight      pre_type          post_type
19       15233      10536      18  cb_intrinsic  descending_neuron
34       15233      10197      12  cb_intrinsic  descending_neuron
45       15233      10135       8  cb_intrinsic  descending_neuron
165      15233      10739       3  cb_intrinsic  descending_neuron
1243     15233      10244       1  cb_intrinsic  descending_neuron


> these five-digit IDs are clustered in the same narrow range as 10001, which was `DNp01` (the Giant Fiber)

In [60]:
# just checking the body_post to see what they are
descending_ids = descending_out['body_post'].tolist()
print(traced[traced['bodyId'].isin(descending_ids)][['bodyId', 'type', 'hemibrainType', 'synonyms']])

     bodyId     type hemibrainType               synonyms
123   10135  DNge031           NaN                    NaN
181   10197    DNp02         DNp02       Matsuo 2016: P3b
226   10244    DNp55        WED180                    NaN
503   10536    DNg29           NaN                    NaN
694   10739    DNg24           NaN  Matsuo 2016: AMMC-Db7


> two out of five come back with real names attached:
> - `DNp02`, synonym `Matsuo 2016: P3b`
> - `DNg24`, synonym `Matsuo 2016: AMMC-Db7`, a weaker connection but also real
>
> `DNge031`, `DNg29`, `WED180` have no information or publication yet
>
> the `AMMC` in the `DNg24` synonym points at the antennal mechanosensory and motor center, where Johnston's organ neurons arrive, so both look auditory-adjacent
>
> note: published morphology work (Namiki et al. 2018, eLife) places `DNp02` neurites in both the AMMC and the LC4 visual glomerulus, so it may not be purely auditory

#### Hop 3: DNp02, descending_neuron

knowing about `DNp02`, that will be used (`bodyId` 10197)

In [61]:
dnp02_id = 10197

filter_expr = (pc.field("body_pre") == dnp02_id) | (pc.field("body_post") == dnp02_id)
dnp02_neighbours = dataset.to_table(filter=filter_expr).to_pandas()

dnp02_neighbours['pre_type'] = dnp02_neighbours['body_pre'].map(id_to_superclass)
dnp02_neighbours['post_type'] = dnp02_neighbours['body_post'].map(id_to_superclass)

outgoing = dnp02_neighbours[dnp02_neighbours['body_pre'] == dnp02_id]
print(outgoing.shape)

weighted_out = (
    outgoing.groupby('post_type', dropna=False)['weight']
    .sum()
    .sort_values(ascending=False)
)
print(weighted_out)

(6945, 5)
post_type
NaN                   9482
vnc_intrinsic         2510
ascending_neuron      1021
descending_neuron      784
cb_intrinsic           765
vnc_motor               50
visual_centrifugal      38
vnc_efferent            15
efferent_ascending       8
cb_motor                 3
sensory_descending       1
visual_projection        1
vnc_sensory              1
Name: weight, dtype: int64


> there exists a connection to `vnc_motor`! weight 50
>
> however, total outgoing weight here is roughly 14,679, and `vnc_motor` is 50 of that, compare that to `vnc_intrinsic` at 2510. the tiny direct `vnc_motor` link might be a real but minor shortcut, while the "real" pathway is likely one more hop through `vnc_intrinsic`
>
> the NaN bucket is now 9482, `DNp02` is clearly a major hub (6945 outgoing edges vs 285 total neighbour rows for `JO-B1_a`), so a bigger unannotated tail isn't shocking, but not shocking isn't the same as being checked. so that will be checked:

In [62]:
nan_out = outgoing[outgoing['post_type'].isna()]
nan_ids = nan_out['body_post'].unique()
print(len(nan_ids), 'unique nan targets')
print(df[df['bodyId'].isin(nan_ids)]['status'].value_counts())

6369 unique nan targets
status
Orphan         2
Glia           2
Unimportant    2
Name: count, dtype: int64


> 6369 unique NaN targets, and only 6 of them exist in `df` at all (2 each: orphan, glia, unimportant)
>
> NaN weight really is just signal going into the unfinished part of the segmentation, so the weight check will be skipped

In [63]:
# identify the actual vnc_motor neurons that were hit directly
motor_targets = outgoing[outgoing['post_type'] == 'vnc_motor'].sort_values('weight', ascending=False)
print(traced[traced['bodyId'].isin(motor_targets['body_post'])][['bodyId', 'type', 'instance', 'somaNeuromere']])

        bodyId                         type                       instance  \
169826  800911               Ti extensor MN               Ti extensor MN_L   
170052  801220                        i1 MN                        i1 MN_L   
170856  802295                 Ti flexor MN                 Ti flexor MN_L   
171767  803480                       MNml82                       MNml82_L   
172695  804642                         TTMn                         TTMn_L   
173368  805477               Tr extensor MN               Tr extensor MN_L   
186385  832490  Sternal anterior rotator MN  Sternal anterior rotator MN_L   
187898  904657          Sternotrochanter MN          Sternotrochanter MN_L   
190952  924167                        STTMm                        STTMm_L   

       somaNeuromere  
169826            T2  
170052            T2  
170856            T2  
171767            T2  
172695            T2  
173368            T2  
186385            T2  
187898            T2  
190952      

> named leg muscle motor neurons were hit, and one jumps out immediately: `TTMn`, the tergotrochanteral motor neuron, the same effector neuron driven by `DNp01` (the Giant Fiber) to trigger a fly's jump-takeoff escape response, which is the very first neuron looked at
>
> so `DNp02` also drives that same jump muscle, alongside genuine leg extensor and leg flexor motor neurons
>
> every single one of these sits at `somaNeuromere == T2`. `T1`, `T2` and `T3` are the three segments of the fly's middle section (the thorax), one for each pair of legs, front to back:
> - `T1`: the front pair of legs
> - `T2`: the middle pair of legs, which push off when the fly jumps
> - `T3`: the back pair of legs
>
> <div align="center">
>  <img src="../img/2.jpg" alt="fly legs" width="40%">
> </div>
>
> so these motor neurons have their cell bodies in the middle-leg segment

#### Side track: vnc_intrinsic

the largest weights that `DNp02` connects to are `vnc_intrinsic`, so look at those too:

In [64]:
vnc_intrinsic_targets = outgoing[outgoing['post_type'] == 'vnc_intrinsic'].sort_values('weight', ascending=False)
print(vnc_intrinsic_targets.shape)
print(vnc_intrinsic_targets.head(10))

(260, 5)
    body_pre  body_post  weight           pre_type      post_type
9      10197     800373     120  descending_neuron  vnc_intrinsic
16     10197     907745      86  descending_neuron  vnc_intrinsic
28     10197     902275      68  descending_neuron  vnc_intrinsic
32     10197     800136      65  descending_neuron  vnc_intrinsic
35     10197     812084      63  descending_neuron  vnc_intrinsic
40     10197     801689      59  descending_neuron  vnc_intrinsic
45     10197     913441      55  descending_neuron  vnc_intrinsic
48     10197     919595      53  descending_neuron  vnc_intrinsic
69     10197     940542      43  descending_neuron  vnc_intrinsic
81     10197     801280      40  descending_neuron  vnc_intrinsic


In [65]:
# don't stop at the superclass label, look at the actual named identities of the strongest few
top_vnc_ids = vnc_intrinsic_targets.head(10)['body_post']
print(traced[traced['bodyId'].isin(top_vnc_ids)][['bodyId', 'type', 'instance', 'somaNeuromere', 'subclass']])

        bodyId      type    instance somaNeuromere subclass
169254  800136  IN06B035  IN06B035_R            T2       BI
169419  800373  IN23B001  IN23B001_L            T1       BI
170103  801280  IN06B021  IN06B021_L            T1       BI
170415  801689  IN00A010  IN00A010_M            T1       BI
178817  812084  IN06B028  IN06B028_R            T1       BI
187051  902275  IN06B018  IN06B018_R            T1       BI
188858  907745  IN00A030  IN00A030_M            T1       BI
190449  913441  IN05B032  IN05B032_L            T2       BR
190795  919595  IN00A010  IN00A010_M            T2       BI
191672  940542  IN11A020  IN11A020_L            T1      NaN


> every single one of these is named `IN___`, this is where the real processing muscle is, not the small direct shortcut
>
> `IN00A010` shows up twice in the top 10, once in `T1` and once in `T2`. are they the same neuron recorded twice, or repeated copies?

In [66]:
# IN00A010 shows up twice, once in T1 (801689) and once in T2 (919595)
print(traced[traced['bodyId'].isin([801689, 919595])][['bodyId', 'type', 'serialMotif', 'mancSerial']])

        bodyId      type     serialMotif  mancSerial
170415  801689  IN00A010  vnc convergent     12565.0
190795  919595  IN00A010  vnc convergent     12565.0


> `IN00A010` appears in both `T1` and `T2` with the same `serialMotif` (`vnc convergent`) and the same `mancSerial` (12565), so they look like members of one repeating set, one copy per segment, not duplicate records of a single neuron, since a duplicate wouldn't have its cell body in two different segments

moving on, in the output prior to `IN00A010` showing up twice, `DNp02` reaches `T2` motor neurons directly, but most of the interneurons it talks to are in `T1`: 7 of the 10 `vnc_intrinsic_targets` sit in `T1` (front legs), only 3 in `T2` (middle legs). so the link through `vnc_intrinsic` might drive the front legs instead.

see the `IN23B001` check below, where a `T1` neuron drives `T2` and `T3` motor neurons:


In [67]:
in23b001_id = 800373  # strongest single connection, weight 120, T1
filter_expr = (pc.field("body_pre") == in23b001_id) | (pc.field("body_post") == in23b001_id)
in_neighbours = dataset.to_table(filter=filter_expr).to_pandas()
in_neighbours['post_type'] = in_neighbours['body_post'].map(id_to_superclass)

out_in = in_neighbours[in_neighbours['body_pre'] == in23b001_id]
print(out_in.groupby('post_type', dropna=False)['weight'].sum().sort_values(ascending=False))

post_type
NaN                   9639
vnc_intrinsic         3674
ascending_neuron       374
vnc_motor              225
descending_neuron       61
sensory_descending       1
Name: weight, dtype: int64


> `IN23B001` (the T1 branch just checked) drives `vnc_motor` directly, weight 225
>
> that's over 4x stronger than the 50-weight direct motor link found from `DNp02`. `DNp02` mostly drives interneurons, and those interneurons are where the strong, reliable motor drive actually comes from, the direct `DNp02` to motor link was the minor shortcut, and this interneuron layer is the main event

In [68]:
# which specific motor neurons IN23B001 lands on, same move as before
motor_from_in = out_in[out_in['post_type'] == 'vnc_motor']
print(traced[traced['bodyId'].isin(motor_from_in['body_post'])][['bodyId', 'type', 'instance', 'somaNeuromere']])

            bodyId                         type  \
169593      800589                 Tr flexor MN   
169783      800851          Sternotrochanter MN   
170177      801375          Sternotrochanter MN   
170311      801548  Sternal anterior rotator MN   
170438      801721                 Tr flexor MN   
170448      801736                       MNhl62   
170511      801813  Sternal anterior rotator MN   
170608      801946  Sternal anterior rotator MN   
170812      802232                 Tr flexor MN   
170856      802295                 Ti flexor MN   
171445      803072                 Ti flexor MN   
171504      803147                 Ti flexor MN   
172240      804078                       MNhl29   
172280      804131                       hg2 MN   
172918      804919                 Ti flexor MN   
172989      805010                 Ti flexor MN   
176056      808685                 Ti flexor MN   
181816      815958            Acc. tr flexor MN   
184532      821449             

> worth noting: no T1 targets at all, despite `IN23B001` itself sitting in T1
>
> instead it's split between T2 (13) and T3 (19), the middle and hind legs
>
> this lines up with the `serialMotif` label from before, `vnc convergent`, convergent is the right word for a hub that gathers/distributes across segments rather than staying local

#### The two pathways

> two full, verified, named pathways, both starting from the same auditory input:
> - direct link (weak, weight 50): `JO-B1_a -> 15233 -> DNp02 -> TTMn/extensor/flexor motor neurons`, all in T2 (middle leg)
> - via an interneuron (strong, weight 225): `JO-B1_a -> 15233 -> DNp02 -> IN23B001 -> motor neurons`, `IN23B001` sits in T1 but its motor targets are in T2 and T3 (middle and hind legs)
>
> the second path is the stronger, more biologically convincing one: sound drives an interneuron layer, which drives motor output, the actual expected shape of a reflex circuit, not just a lucky direct wire

#### Building the graph

this gives a complete, hand-verified, four-hop pathway with a real biological shape. the weights below were typed in by hand from the printouts above

In [69]:
import networkx as nx

G = nx.DiGraph()

confirmed_path_edges = [
    (jo_id, 15233, 46),
    (15233, 10197, 12),        # DNp02
    (10197, 800373, 120),      # IN23B001
]

for pre, post, w in confirmed_path_edges:
    G.add_edge(pre, post, weight=w)

for _, row in motor_from_in.iterrows():
    G.add_edge(in23b001_id, row['body_post'], weight=row['weight'])

print(G.number_of_nodes(), 'nodes,', G.number_of_edges(), 'edges')
print(list(nx.all_simple_paths(G, source=jo_id, target=800589)))  # one example flexor MN

37 nodes, 36 edges
[[np.int64(42619), 15233, 10197, 800373, 800589]]


### Conclusion

what this file actually established:
- audio input does exist in the data: the antennal nerve carries both smell and hearing, and within it only the `JO-A`/`JO-B` subtypes (138 neurons) are sound sensors. `JO-B` (88 vs 50) is the better candidate for a rhythm-based input, since it's wired toward the song-relay pathway
- a hand-built, four-hop pathway exists from a fully-resolved `JO-B` neuron (`JO-B1_a`) to leg motor neurons, through `15233` and `DNp02`, either by a weak direct link (weight 50, including `TTMn`, the jump motor neuron the Giant Fiber also drives) or by a much stronger route through the interneuron `IN23B001` (weight 225), landing in the middle and hind legs
- checking both ends of the visual path caught a mistake: the endpoint used in `03` (`164190`) is an abdominal motor neuron, not a leg one, so the timing results there describe a path to the wrong body part, even though the mechanics themselves still hold

what's still open:
- this pathway is structural only, nothing has been simulated through it yet, and its weights were typed in by hand
- `DNp02` may not be purely auditory, published work places it in a visual glomerulus too (Namiki et al. 2018)

so next:
- run the scaled LIF from `03` on this audio path
- redo the visual path in 02 with a leg motor neuron as the target (the commented-out code above), then rerun 03 on it